In [1]:
import json
import re
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ============================================================
# 1. Configuration
# ============================================================

from pathlib import Path

DATA_PATH = Path("/home/sudhanshu_bajpai/projects/Voice_assistant/data/intents.json")
MODEL_PATH = Path("/home/sudhanshu_bajpai/projects/Voice_assistant/data/intent_model.keras")
MAX_LEN = 10
EMBEDDING_DIM = 128
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 0.001


# ============================================================
# 2. Load Dataset
# ============================================================

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Loaded intents:", len(data))


# ============================================================
# 3. Tokenizer
# ============================================================

def tokenize(text):
    return re.findall(
        r"[a-zA-Z0-9']+",
        text.lower()
    )


# ============================================================
# 4. Build Vocabulary
# ============================================================

all_words = set()

for examples in data.values():

    for sentence in examples:

        tokens = tokenize(sentence)

        all_words.update(tokens)


PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

vocab = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}

for word in sorted(all_words):

    vocab[word] = len(vocab)


print("Vocabulary size:", len(vocab))


# ============================================================
# 5. Intent Labels
# ============================================================

intents = list(data.keys())

intent_to_id = {
    intent: i
    for i, intent in enumerate(intents)
}

print("\nIntents:")

for intent, index in intent_to_id.items():

    print(index, "->", intent)


# ============================================================
# 6. Encode Sentence
# ============================================================

def encode_sentence(sentence):

    tokens = tokenize(sentence)

    ids = []

    for token in tokens:

        word_id = vocab.get(
            token,
            vocab[UNK_TOKEN]
        )

        ids.append(word_id)

    # Truncate
    ids = ids[:MAX_LEN]

    # Padding
    while len(ids) < MAX_LEN:

        ids.append(vocab[PAD_TOKEN])

    return ids


# ============================================================
# 7. Create Samples
# ============================================================

samples = []

for intent, examples in data.items():

    label = intent_to_id[intent]

    for sentence in examples:

        encoded = encode_sentence(sentence)

        samples.append(
            (
                encoded,
                label
            )
        )


# Shuffle
random.shuffle(samples)


# ============================================================
# 8. Convert to NumPy
# ============================================================

X = np.array(
    [sample[0] for sample in samples],
    dtype=np.int32
)

y = np.array(
    [sample[1] for sample in samples],
    dtype=np.int32
)

print("\nDataset shape:")
print("X:", X.shape)
print("y:", y.shape)


# ============================================================
# 9. Train / Validation Split
# ============================================================

split = int(
    0.8 * len(X)
)

X_train = X[:split]
y_train = y[:split]

X_val = X[split:]
y_val = y[split:]


print("\nTraining samples:", len(X_train))
print("Validation samples:", len(X_val))


# ============================================================
# 10. Build TensorFlow Dataset
# ============================================================

train_dataset = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)
)

train_dataset = (
    train_dataset
    .shuffle(len(X_train))
    .batch(BATCH_SIZE)
)


val_dataset = tf.data.Dataset.from_tensor_slices(
    (X_val, y_val)
)

val_dataset = val_dataset.batch(
    BATCH_SIZE
)


# ============================================================
# 11. Build Neural Network
# ============================================================

model = keras.Sequential([

    # Input
    layers.Input(
        shape=(MAX_LEN,)
    ),

    # Word embeddings
    layers.Embedding(
        input_dim=len(vocab),
        output_dim=EMBEDDING_DIM,
        mask_zero=True
    ),

    # Average word vectors
    layers.GlobalAveragePooling1D(),

    # Hidden layer
    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dense(
        64,
        activation="relu"
    ),

    # Second hidden layer
    layers.Dense(
        32,
        activation="relu"
    ),

    # Output layer
    layers.Dense(
        len(intents),
        activation="softmax"
    )
])


# ============================================================
# 12. Display Architecture
# ============================================================

print("\nModel Architecture:\n")

model.summary()


# ============================================================
# 13. Compile Model
# ============================================================

optimizer = keras.optimizers.Adam(
    learning_rate=LEARNING_RATE
)

model.compile(

    optimizer=optimizer,

    loss=(
        "sparse_categorical_crossentropy"
    ),

    metrics=[
        "accuracy"
    ]
)


# ============================================================
# 14. Train
# ============================================================

print("\nStarting training...\n")

history = model.fit(

    train_dataset,

    validation_data=val_dataset,

    epochs=EPOCHS
)


# ============================================================
# 15. Final Evaluation
# ============================================================

loss, accuracy = model.evaluate(
    val_dataset,
    verbose=0
)

print(
    f"\nValidation Loss: {loss:.4f}"
)

print(
    f"Validation Accuracy: {accuracy:.4f}"
)


# ============================================================
# 16. Save Model
# ============================================================

model.save(
    MODEL_PATH
)

print(
    "\nModel saved to:",
    MODEL_PATH
)


# ============================================================
# 17. Save Vocabulary + Intents
# ============================================================

metadata_path = ("/home/sudhanshu_bajpai/projects/Voice_assistant/data/intent_metadata.json")

metadata = {

    "vocab": vocab,

    "intents": intents,

    "max_len": MAX_LEN
}


with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        metadata,
        f,
        indent=4
    )


print(
    "Metadata saved to:",
    metadata_path
)

I0000 00:00:1790925773.847262    5135 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790925774.153833    5135 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790925776.382296    5135 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Loaded intents: 10
Vocabulary size: 113

Intents:
0 -> OPEN_EDGE
1 -> OPEN_VSCODE
2 -> OPEN_NOTEPAD
3 -> CHECK_GPU
4 -> CHECK_CPU
5 -> OPEN_PROJECT
6 -> START_JUPYTER
7 -> STOP_ASSISTANT
8 -> SEARCH_WEB
9 -> MAKE_NOTES

Dataset shape:
X: (178, 10)
y: (178,)

Training samples: 142
Validation samples: 36


W0000 00:00:1790925781.774897    5135 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1790925781.780024    5135 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1790925782.008988    5135 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5262 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a



Model Architecture:



Model: "sequential"

...


Starting training...

Epoch 1/100


/home/sudhanshu_bajpai/tf_env/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1790925783.685104    5257 service.cc:153] XLA service 0x7cb1c8045580 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1790925783.685136    5257 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5050 Laptop GPU, Compute Capability 12.0a (Driver: 13.1.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1790925783.705565    5257 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1790925783.858517    5257 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1790925783.865479    5257 dot_merger.cc

1/5 ━━━━━━━━━━━━━━━━━━━━ 15s 4s/step - accuracy: 0.1250 - loss: 2.3033

I0000 00:00:1790925786.783386    5257 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
I0000 00:00:1790925786.966827    5257 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1769__.20
I0000 00:00:1790925787.737593    5479 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_20', 8 bytes spill stores, 8 bytes spill loads



5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.1549 - loss: 2.3011 - val_accuracy: 0.1389 - val_loss: 2.2907
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3310 - loss: 2.2816 - val_accuracy: 0.1389 - val_loss: 2.2757
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3028 - loss: 2.2574 - val_accuracy: 0.1667 - val_loss: 2.2526
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3310 - loss: 2.2200 - val_accuracy: 0.1667 - val_loss: 2.2144
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.4085 - loss: 2.1627 - val_accuracy: 0.1944 - val_loss: 2.1538
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4437 - loss: 2.0763 - val_accuracy: 0.2222 - val_loss: 2.0693
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4789 - loss: 1.9515 - val_accuracy: 0.2500 - val_loss: 1.9567
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5493 - loss: 1.7841 - val_accuracy: 0.2778 - val_loss: 1.8326
Epoch 9/100
